In [ ]:
# WVS Data Preprocessing with Ollama
# This notebook preprocesses the World Values Survey trend file by:
# 1. Loading the dataset
# 2. Decoding encrypted column names using documentation
# 3. Using Ollama for intelligent preprocessing
# 4. Displaying the first 10 rows of processed data

In [1]:
# Cell 1: Import Required Libraries

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import json
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# For Ollama integration
import subprocess
import sys

In [2]:
# Cell 2: Setup Ollama Connection

def check_ollama_running():
    """Check if Ollama is running and available"""
    try:
        result = subprocess.run(['ollama', 'list'], 
                              capture_output=True, 
                              text=True, 
                              timeout=10)
        return result.returncode == 0
    except:
        return False

def query_ollama(prompt, model="llama2"):
    """Send a query to Ollama and get response"""
    try:
        result = subprocess.run(
            ['ollama', 'run', model, prompt],
            capture_output=True,
            text=True,
            timeout=30
        )
        return result.stdout.strip()
    except Exception as e:
        print(f"Error querying Ollama: {e}")
        return None

print("Checking Ollama connection...")
if check_ollama_running():
    print("✅ Ollama is running")
else:
    print("⚠️  Ollama is not running. Please start Ollama first:")
    print("   Run: ollama serve")
    print("   Then: ollama pull llama2")

Checking Ollama connection...
✅ Ollama is running


In [3]:
# Cell 3: Load the Dataset

def load_wvs_data(file_path="EVS_WVS_Trend_File.dta"):
    """
    Load WVS trend file (assuming Stata format)
    Adjust file extension based on your actual data file
    """
    try:
        # Try different formats based on file extension
        if file_path.endswith('.dta'):
            df = pd.read_stata(file_path)
        elif file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        elif file_path.endswith('.sav'):
            import pyreadstat
            df, meta = pyreadstat.read_sav(file_path)
        else:
            df = pd.read_csv(file_path)
        
        print(f"✅ Dataset loaded successfully")
        print(f"   Shape: {df.shape}")
        print(f"   Columns: {len(df.columns)}")
        return df
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        return None

# Load the data - adjust path as needed
df = load_wvs_data("EVS_WVS_Trend_File.dta")

if df is None:
    # If file not found, create sample data for demonstration
    print("Creating sample WVS data for demonstration...")
    np.random.seed(42)
    n_rows = 1000
    
    sample_data = {
        'S002': np.random.choice(range(1, 50), n_rows),  # Country code
        'S003': np.random.choice(range(1960, 2023), n_rows),  # Year
        'S020': np.random.choice(range(1, 100), n_rows),  # Wave
        'X001': np.random.choice([1, 2], n_rows),  # Gender
        'X003': np.random.randint(18, 90, n_rows),  # Age
        'A001': np.random.choice([1, 2, 3, 4], n_rows),  # Important in life: family
        'A002': np.random.choice([1, 2, 3, 4], n_rows),  # Important in life: friends
        'F114': np.random.choice([1, 2, 3, 4, 5], n_rows),  # Satisfaction with life
        'E069': np.random.choice([1, 2, 3, 4], n_rows),  # Confidence: churches
        'E069_01': np.random.choice([1, 2, 3, 4], n_rows),  # Confidence: armed forces
    }
    
    df = pd.DataFrame(sample_data)
    print(f"✅ Sample data created with shape: {df.shape}")

❌ Error loading dataset: [Errno 2] No such file or directory: 'EVS_WVS_Trend_File.dta'
Creating sample WVS data for demonstration...
✅ Sample data created with shape: (1000, 10)
